# 04a — ReAct Agent + Tool Use
**Domain:** IT / GitHub Issues &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** ReAct pattern, function calling, tool loop, guardrails

---
### ReAct Pattern (Reason + Act)
The agent alternates between **Thought → Action → Observation** until it produces a final answer.  
Each iteration the LLM decides which tool to call; the result feeds back as an observation.


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ llm_checker loaded")


In [ ]:
import pandas as pd

DATA_PATH = os.path.join("data", "github_issues", "github_issues.csv")
if os.path.exists(DATA_PATH):
    issues_df = pd.read_csv(DATA_PATH)
    print(f"✅ Loaded {len(issues_df)} GitHub issues — columns: {issues_df.columns.tolist()}")
else:
    print("⚠️  Run setup_datasets.py first. Using synthetic fallback.")
    issues_df = pd.DataFrame({
        "title": ["API broken on POST /users", "Memory leak in worker", "Login fails after timeout",
                  "Rate limiting not working", "DB connection pool exhausted"],
        "body":  ["Returns 500", "OOM on large input", "JWT validation error",
                  "1000 req/s allowed", "Max connections reached"],
        "state": ["open", "open", "closed", "open", "closed"],
    })


---
## 🔵 Example — Ex 04a-1: Agent with 2 Tools

In [ ]:
from openai import OpenAI
import json

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# ── Tool implementations ───────────────────────────────────────────────────
def search_issues(query: str) -> list:
    mask = issues_df.apply(lambda r: query.lower() in str(r).lower(), axis=1)
    return issues_df[mask].head(3)[["title", "state"]].to_dict("records")

def get_issue_details(issue_id: int) -> dict:
    if 0 <= issue_id < len(issues_df):
        return issues_df.iloc[issue_id].to_dict()
    return {"error": "Not found"}

TOOL_MAP = {"search_issues": search_issues, "get_issue_details": get_issue_details}

TOOLS = [
    {"type": "function", "function": {
        "name": "search_issues",
        "description": "Search GitHub issues by keyword",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_issue_details",
        "description": "Get full details of a GitHub issue by row ID",
        "parameters": {"type": "object",
                       "properties": {"issue_id": {"type": "integer"}},
                       "required": ["issue_id"]}}},
]

# ── ReAct loop ─────────────────────────────────────────────────────────────
def run_agent(user_query: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content": "You are an IT support agent. Use tools to answer."},
        {"role": "user",   "content": user_query},
    ]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model="local-model", messages=messages, tools=TOOLS, tool_choice="auto"
        )
        msg = resp.choices[0].message
        messages.append(msg)
        if msg.tool_calls:
            for tc in msg.tool_calls:
                fn   = TOOL_MAP.get(tc.function.name, lambda **kw: {"error": "unknown"})
                args = json.loads(tc.function.arguments)
                result = fn(**args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                  "content": json.dumps(result)})
        else:
            return msg.content
    return "Max steps reached."

answer = run_agent("Find open issues related to memory or connection problems")
print(answer)


---
## 🟡 Exercise — Ex 04a-2: Incident Response Agent with 4 Tools

In [ ]:
show_exercise(
    "04a-2", "Incident response agent with 4 tools",
    """Add tools: run_diagnostics(service_name), create_ticket(title, severity),
check_logs(service_name, hours_back).
Build incident_agent(query) that uses >= 3 different tools per run.
The loop must terminate reliably (finish_reason == "stop").""",
    hints=[
        "Each tool: Python function + JSON schema dict (same pattern as Ex 04a-1)",
        "Severity: 1=critical, 2=high, 3=medium, 4=low",
        "Break the loop when resp.choices[0].finish_reason == 'stop'",
    ],
    checks=[
        "incident_agent() returns a string (terminates)",
        "INCIDENT_TOOLS list has >= 4 entries",
        "At least 3 different tool names called in one run",
    ],
)


In [ ]:
import random, datetime

# ── Simulated tool implementations ─────────────────────────────────────────
def run_diagnostics(service_name: str) -> dict:
    """Run live diagnostics on a service."""
    return {
        "service": service_name,
        "cpu_pct": random.randint(10, 95),
        "memory_pct": random.randint(20, 90),
        "status": random.choice(["healthy", "degraded", "down"]),
    }

def create_ticket(title: str, severity: int) -> dict:
    """Create an incident ticket; returns ticket_id."""
    return {"ticket_id": f"INC-{random.randint(1000, 9999)}", "title": title, "severity": severity}

def check_logs(service_name: str, hours_back: int = 1) -> list:
    """Return recent log entries for a service."""
    return [
        f"[{datetime.datetime.now()}] {service_name}: ERROR connection timeout",
        f"[{datetime.datetime.now()}] {service_name}: WARN  latency {random.randint(200, 2000)}ms",
    ]

# TODO: define INCIDENT_TOOLS — list of 4 tool schema dicts
INCIDENT_TOOLS = []  # ← fill in

# TODO: implement incident_agent using the 4 tools + original 2
def incident_agent(query: str) -> str:
    pass


In [ ]:
tools_called = []   # track which tools were invoked

try:
    answer_04a2 = incident_agent("The payment service is slow and timing out. Diagnose and file a ticket.")
except Exception as e:
    answer_04a2 = None
    print(f"Error: {e}")

check([
    (len(INCIDENT_TOOLS) >= 4,        f"INCIDENT_TOOLS has >= 4 entries (got {len(INCIDENT_TOOLS)})"),
    (answer_04a2 is not None,         "incident_agent() returns a result"),
    (isinstance(answer_04a2, str),    "Returns a string"),
], exercise_id="04a-2")


In [ ]:
evaluate(incident_agent,
         "Incident response ReAct agent with 4 tools: search_issues, run_diagnostics, check_logs, create_ticket.",
         exercise_id="04a-2")


---
## 🔴 Challenge — Ex 04a-3: Guardrails & max_steps

In [ ]:
show_exercise(
    "04a-3", "Guardrails and max_steps",
    """Extend incident_agent with:
1. max_steps=8 hard limit that terminates the loop.
2. BLOCKED_TOOLS list — any call to a blocked tool returns a guardrail error instead of executing.
3. Retry on tool exception (max 2 retries per tool call).
Demonstrate that blocked tools are rejected and max_steps stops the loop.""",
    hints=[
        "Before executing: if tc.function.name in BLOCKED_TOOLS: return error dict",
        "Wrap tool execution in try/except; retry up to 2 times",
        "Pass max_steps as a parameter so tests can set it to 1",
    ],
    checks=[
        "BLOCKED_TOOLS blocks execution and inserts an error observation",
        "max_steps terminates the loop without raising an exception",
        "Tool exception triggers retry (visible via counter)",
    ],
)


In [ ]:
BLOCKED_TOOLS = ["delete_repository", "shutdown_cluster"]

def safe_incident_agent(query: str, max_steps: int = 8) -> str:
    """Incident agent with guardrails: blocked tools, max_steps, retry."""
    # TODO: copy incident_agent and add the three guardrails
    pass


# ── Quick smoke tests ──────────────────────────────────────────────────────
result_normal  = safe_incident_agent("Check database health and create a ticket.")
result_maxstep = safe_incident_agent("Check health.", max_steps=1)   # must not loop forever

print("Normal  :", result_normal[:120] if result_normal else None)
print("MaxStep :", result_maxstep[:120] if result_maxstep else None)

check([
    (result_normal  is not None, "safe_incident_agent() returns result"),
    (result_maxstep is not None, "Terminates even with max_steps=1"),
], exercise_id="04a-3")


In [ ]:
check([
    (len(INCIDENT_TOOLS) >= 4,        "4 tools defined"),
    (incident_agent("test") is not None, "incident_agent works"),
    (safe_incident_agent("test") is not None, "Guardrail agent works"),
], exercise_id="04a-final")
progress()
